# 10 - API 网关层内部分析

**目标**: 测量 API 网关层每个环节的时间开销，量化路由、中间件、GraphQL 解析、请求转发的成本。

**覆盖事件链**:
```
[API-1] Nginx 代理转发开销（生产环境）
[API-2] FastAPI 框架路由匹配开销
[API-3] CORS 中间件处理开销
[API-4] GraphQL 解析 + 校验开销（vs REST）
[API-5] API Gateway → AI Engine 转发开销（内部 HTTP hop）
[API-6] 并发请求队列时间
```

**对应服务**:
- `xiaocai-api` (port 8001): FastAPI + Strawberry GraphQL
- `xiaocai-ai-engine` (port 8002): FastAPI + LangGraph
- Nginx (port 80/443): 生产环境反向代理（本 notebook 分析其配置开销）

In [ ]:
# ============================================================
# CELL 0: 配置区域 — 所有可控参数在此修改
# ============================================================

# --- 服务端点 ---
API_GATEWAY_URL  = "http://localhost:8001"    # xiaocai-api (FastAPI + GraphQL)
AI_ENGINE_URL    = "http://localhost:8002"    # xiaocai-ai-engine (直接)

# --- 采集参数 ---
N_SAMPLES        = 20     # 每项测试采集次数
WARMUP_ROUNDS    = 3      # 预热轮数（不计入统计）
TIMEOUT_S        = 15.0   # 单次请求超时（秒）

# --- 并发测试参数 ---
CONCURRENCY_LEVELS = [1, 5, 10, 20]  # 并发数阶梯
CONCURRENT_N       = 30              # 每个并发级别总请求数

# --- 输出路径 ---
OUTPUT_DIR = "../data"

# --- GraphQL 查询定义 ---
GQL_SIMPLE_HELLO = """
{ hello }
"""

GQL_INTROSPECTION = """
{
  __schema {
    types {
      name
      kind
    }
  }
}
"""

GQL_MUTATION_CHAT = """
mutation Chat($message: String!, $sessionId: String) {
  chat(message: $message, sessionId: $sessionId) {
    reply
  }
}
"""

# --- 测试消息（不触发完整 LLM 调用的轻量消息）---
TEST_MESSAGE = "hello"

print("配置加载完成")
print(f"  API Gateway : {API_GATEWAY_URL}")
print(f"  AI Engine   : {AI_ENGINE_URL}")
print(f"  采样次数    : {N_SAMPLES}")
print(f"  并发级别    : {CONCURRENCY_LEVELS}")

In [ ]:
# ============================================================
# CELL 1: 依赖导入
# ============================================================

import httpx
import asyncio
import time
import json
import os
import subprocess
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

# 创建输出目录
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print("依赖加载完成")

In [ ]:
# ============================================================
# CELL 2: 核心测量工具函数
# ============================================================

def measure_get(url: str, n: int = N_SAMPLES, warmup: int = WARMUP_ROUNDS) -> List[float]:
    """同步 GET 请求延迟测量（ms）"""
    results = []
    for i in range(n + warmup):
        try:
            t0 = time.perf_counter()
            with httpx.Client(timeout=TIMEOUT_S) as client:
                r = client.get(url)
            elapsed = (time.perf_counter() - t0) * 1000
            if i >= warmup:
                results.append(elapsed)
        except Exception as e:
            if i >= warmup:
                print(f"  GET {url} 失败: {e}")
    return results


def measure_post_json(url: str, payload: dict, n: int = N_SAMPLES,
                      warmup: int = WARMUP_ROUNDS) -> List[float]:
    """同步 POST JSON 请求延迟测量（ms）"""
    results = []
    for i in range(n + warmup):
        try:
            t0 = time.perf_counter()
            with httpx.Client(timeout=TIMEOUT_S) as client:
                r = client.post(url, json=payload)
            elapsed = (time.perf_counter() - t0) * 1000
            if i >= warmup:
                results.append(elapsed)
        except Exception as e:
            if i >= warmup:
                print(f"  POST {url} 失败: {e}")
    return results


def measure_options(url: str, origin: str = "http://localhost:3001",
                    n: int = N_SAMPLES, warmup: int = WARMUP_ROUNDS) -> List[float]:
    """OPTIONS preflight 请求测量（CORS 中间件）"""
    results = []
    headers = {
        "Origin": origin,
        "Access-Control-Request-Method": "POST",
        "Access-Control-Request-Headers": "content-type",
    }
    for i in range(n + warmup):
        try:
            t0 = time.perf_counter()
            with httpx.Client(timeout=TIMEOUT_S) as client:
                r = client.options(url, headers=headers)
            elapsed = (time.perf_counter() - t0) * 1000
            if i >= warmup:
                results.append(elapsed)
        except Exception as e:
            if i >= warmup:
                print(f"  OPTIONS {url} 失败: {e}")
    return results


async def concurrent_requests(url: str, payload: dict, concurrency: int,
                              total: int) -> Dict:
    """并发 POST 请求，返回统计数据"""
    semaphore = asyncio.Semaphore(concurrency)
    latencies = []

    async def single_request():
        async with semaphore:
            t0 = time.perf_counter()
            try:
                async with httpx.AsyncClient(timeout=TIMEOUT_S) as client:
                    r = await client.post(url, json=payload)
                elapsed = (time.perf_counter() - t0) * 1000
                latencies.append(elapsed)
            except Exception as e:
                latencies.append(None)

    tasks = [single_request() for _ in range(total)]
    t_batch_start = time.perf_counter()
    await asyncio.gather(*tasks)
    t_batch_end = time.perf_counter()

    valid = [l for l in latencies if l is not None]
    return {
        "concurrency": concurrency,
        "total_requests": total,
        "success_count": len(valid),
        "wall_time_ms": (t_batch_end - t_batch_start) * 1000,
        "throughput_rps": len(valid) / (t_batch_end - t_batch_start),
        "mean_ms": np.mean(valid) if valid else None,
        "p50_ms":  np.percentile(valid, 50) if valid else None,
        "p95_ms":  np.percentile(valid, 95) if valid else None,
        "max_ms":  np.max(valid) if valid else None,
    }


def stats(data: List[float]) -> Dict:
    """基础统计"""
    if not data:
        return {"mean": None, "p50": None, "p95": None, "max": None, "min": None}
    return {
        "mean": round(np.mean(data), 2),
        "p50":  round(np.percentile(data, 50), 2),
        "p95":  round(np.percentile(data, 95), 2),
        "max":  round(np.max(data), 2),
        "min":  round(np.min(data), 2),
    }


print("工具函数定义完成")

In [ ]:
# ============================================================
# CELL 3: [API-2] FastAPI 路由开销 — 轻量端点基线
# 测量不同复杂度端点的路由匹配时间
# ============================================================

print("=" * 60)
print("[API-2] FastAPI 路由匹配开销")
print("=" * 60)

routing_endpoints = [
    # (标签, URL, 方法, payload, 说明)
    ("AI Engine /health",  f"{AI_ENGINE_URL}/health",  "GET",  None, "AI Engine 直接 - 无路由开销"),
    ("API GW /health",     f"{API_GATEWAY_URL}/health", "GET",  None, "API 网关 /health - 1层路由"),
    ("API GW /",           f"{API_GATEWAY_URL}/",        "GET",  None, "API 网关根路径 - 1层路由"),
    ("AI Engine /status",  f"{AI_ENGINE_URL}/status",   "GET",  None, "AI Engine /status - 带状态检查"),
]

routing_results = {}
for label, url, method, payload, desc in routing_endpoints:
    print(f"\n  测量: {label}")
    print(f"    {method} {url}")
    print(f"    说明: {desc}")
    try:
        if method == "GET":
            data = measure_get(url)
        else:
            data = measure_post_json(url, payload)
        s = stats(data)
        routing_results[label] = {"data": data, "stats": s, "url": url, "desc": desc}
        print(f"    mean={s['mean']}ms  p95={s['p95']}ms  max={s['max']}ms")
    except Exception as e:
        print(f"    错误: {e}")
        routing_results[label] = {"data": [], "stats": {}, "url": url, "desc": desc}

# 计算 API 网关额外开销
if "AI Engine /health" in routing_results and "API GW /health" in routing_results:
    engine_mean = routing_results["AI Engine /health"]["stats"].get("mean", 0)
    gw_mean     = routing_results["API GW /health"]["stats"].get("mean", 0)
    overhead    = gw_mean - engine_mean
    print(f"\n[结论] API 网关路由额外开销: {overhead:.2f}ms")
    print(f"  (API GW {gw_mean}ms - AI Engine {engine_mean}ms)")
    routing_results["_gateway_overhead_ms"] = overhead

In [ ]:
# ============================================================
# CELL 4: [API-3] CORS 中间件开销
# 测量 OPTIONS preflight 请求 vs 正常 GET 请求的差异
# ============================================================

print("=" * 60)
print("[API-3] CORS 中间件开销")
print("=" * 60)

cors_results = {}

# 正常 GET 请求（无 CORS 检查）
print("\n  GET /health (无 CORS 头)...")
get_data = measure_get(f"{API_GATEWAY_URL}/health")
cors_results["GET /health (no CORS)"] = stats(get_data)
print(f"    mean={cors_results['GET /health (no CORS)']['mean']}ms")

# OPTIONS preflight（触发 CORS 中间件）
print("\n  OPTIONS /graphql (CORS preflight)...")
options_data = measure_options(f"{API_GATEWAY_URL}/graphql")
cors_results["OPTIONS /graphql (CORS)"] = stats(options_data)
print(f"    mean={cors_results['OPTIONS /graphql (CORS)']['mean']}ms")

# 带 Origin 头的 GET（触发 CORS 响应头生成）
print("\n  GET /health (带 Origin 头)...")
origin_data = []
for i in range(N_SAMPLES + WARMUP_ROUNDS):
    try:
        t0 = time.perf_counter()
        with httpx.Client(timeout=TIMEOUT_S) as client:
            r = client.get(
                f"{API_GATEWAY_URL}/health",
                headers={"Origin": "http://localhost:3001"}
            )
        elapsed = (time.perf_counter() - t0) * 1000
        if i >= WARMUP_ROUNDS:
            origin_data.append(elapsed)
    except Exception as e:
        print(f"    错误: {e}")

cors_results["GET /health (with Origin)"] = stats(origin_data)
print(f"    mean={cors_results['GET /health (with Origin)']['mean']}ms")

print("\n[CORS 开销对比]")
for label, s in cors_results.items():
    print(f"  {label:<40} mean={s['mean']}ms  p95={s['p95']}ms")

In [ ]:
# ============================================================
# CELL 5: [API-4] GraphQL 解析开销 vs REST 开销
# 对比 GraphQL schema 解析/校验 vs 纯 REST 路由的时间差
# ============================================================

print("=" * 60)
print("[API-4] GraphQL 解析开销 vs REST 开销")
print("=" * 60)

graphql_results = {}

# 基线: REST /health (纯路由匹配，无解析)
print("\n  REST GET /health (基线)...")
rest_data = measure_get(f"{API_GATEWAY_URL}/health")
graphql_results["REST GET /health"] = {"data": rest_data, "stats": stats(rest_data)}
print(f"    mean={graphql_results['REST GET /health']['stats']['mean']}ms")

# GraphQL: 简单 hello 查询（schema 解析 + 路由到 resolver）
print("\n  GraphQL {hello} (最小查询)...")
gql_hello_data = measure_post_json(
    f"{API_GATEWAY_URL}/graphql",
    {"query": GQL_SIMPLE_HELLO}
)
graphql_results["GraphQL {hello}"] = {"data": gql_hello_data, "stats": stats(gql_hello_data)}
print(f"    mean={graphql_results['GraphQL {hello}']['stats']['mean']}ms")

# GraphQL: 内省查询（完整 schema 遍历，最大解析开销）
print("\n  GraphQL introspection (__schema)...")
gql_intro_data = measure_post_json(
    f"{API_GATEWAY_URL}/graphql",
    {"query": GQL_INTROSPECTION}
)
graphql_results["GraphQL __schema"] = {"data": gql_intro_data, "stats": stats(gql_intro_data)}
print(f"    mean={graphql_results['GraphQL __schema']['stats']['mean']}ms")

# GraphQL: Mutation (涉及 resolver 执行，但 AI Engine 可能不可用时返回错误)
print("\n  GraphQL mutation chat (含 AI Engine 转发)...")
gql_mutation_data = measure_post_json(
    f"{API_GATEWAY_URL}/graphql",
    {
        "query": GQL_MUTATION_CHAT,
        "variables": {"message": TEST_MESSAGE, "sessionId": "benchmark-gw-10"}
    },
    n=min(N_SAMPLES, 5)   # mutation 次数少些（涉及 LLM 调用，时间较长）
)
graphql_results["GraphQL mutation chat"] = {"data": gql_mutation_data, "stats": stats(gql_mutation_data)}
print(f"    mean={graphql_results['GraphQL mutation chat']['stats']['mean']}ms")

# 计算 GraphQL 额外开销
rest_mean  = graphql_results["REST GET /health"]["stats"].get("mean", 0)
hello_mean = graphql_results["GraphQL {hello}"]["stats"].get("mean", 0)
gql_overhead = hello_mean - rest_mean

print(f"\n[结论] GraphQL 框架额外开销: {gql_overhead:.2f}ms")
print(f"  (GraphQL hello {hello_mean}ms - REST /health {rest_mean}ms)")
graphql_results["_gql_overhead_ms"] = gql_overhead

In [ ]:
# ============================================================
# CELL 6: [API-5] 内部 HTTP Hop 开销
# API Gateway (8001) → AI Engine (8002) 转发开销
# 通过比较: 直接访问 AI Engine vs 经过 API 网关访问来计算
# ============================================================

print("=" * 60)
print("[API-5] 内部 HTTP Hop 开销 (API Gateway → AI Engine)")
print("=" * 60)

hop_results = {}

# 直接访问 AI Engine /health
print("\n  直接访问 AI Engine /health...")
direct_engine = measure_get(f"{AI_ENGINE_URL}/health")
hop_results["Direct AI Engine /health"] = stats(direct_engine)
print(f"    mean={hop_results['Direct AI Engine /health']['mean']}ms")

# 通过 API 网关访问 /health（API 网关不转发，自己处理）
print("\n  通过 API Gateway /health...")
via_gw_health = measure_get(f"{API_GATEWAY_URL}/health")
hop_results["Via API GW /health"] = stats(via_gw_health)
print(f"    mean={hop_results['Via API GW /health']['mean']}ms")

# 测量 API 网关到 AI Engine 的 HTTP 转发延迟
# 方法: 分别测量两端的 /health，差值 ≈ 网关内部处理 + 本机 HTTP 往返
direct_mean = hop_results["Direct AI Engine /health"]["mean"]
gw_mean     = hop_results["Via API GW /health"]["mean"]

print("\n  测量本机 HTTP 往返延迟 (loopback)...")
# 额外测量: 直接向 AI Engine 发 POST（无业务逻辑），模拟内部 hop
loopback_data = []
for i in range(N_SAMPLES + WARMUP_ROUNDS):
    t0 = time.perf_counter()
    try:
        with httpx.Client(timeout=TIMEOUT_S) as client:
            r = client.get(f"{AI_ENGINE_URL}/health")
        elapsed = (time.perf_counter() - t0) * 1000
        if i >= WARMUP_ROUNDS:
            loopback_data.append(elapsed)
    except:
        pass

hop_results["Loopback GET /health"] = stats(loopback_data)
print(f"    mean={hop_results['Loopback GET /health']['mean']}ms")

# 估算: API Gateway 转发到 AI Engine 的额外 hop 成本
estimated_hop_cost = hop_results["Loopback GET /health"]["mean"]
print(f"\n[结论] 每次内部 HTTP Hop 估算成本: ~{estimated_hop_cost:.2f}ms")
print(f"  (本机 loopback 延迟，不含业务逻辑)")
print(f"  API Gateway 在请求链中增加 2次 hop (client→GW, GW→Engine)")
print(f"  估算总 hop 开销: ~{estimated_hop_cost * 2:.2f}ms")

hop_results["_estimated_hop_ms"] = estimated_hop_cost
hop_results["_total_hop_overhead_ms"] = estimated_hop_cost * 2

In [ ]:
# ============================================================
# CELL 7: [API-6] 并发请求队列时间
# 测量不同并发级别下的延迟变化（队列等待时间）
# ============================================================

print("=" * 60)
print("[API-6] 并发请求队列时间分析")
print("=" * 60)

concurrency_results = []

# 目标: 轻量的 GET 请求，避免 LLM 带来的不确定性
# 使用 /health 测量纯 FastAPI 并发处理能力

async def run_all_concurrency():
    for c in CONCURRENCY_LEVELS:
        print(f"\n  并发级别: {c} (共 {CONCURRENT_N} 个请求)")
        # 使用 GraphQL hello 查询（有 schema 解析，比 /health 更有代表性）
        result = await concurrent_requests(
            f"{API_GATEWAY_URL}/graphql",
            {"query": GQL_SIMPLE_HELLO},
            concurrency=c,
            total=CONCURRENT_N
        )
        concurrency_results.append(result)
        print(f"    成功: {result['success_count']}/{result['total_requests']}")
        print(f"    吞吐量: {result['throughput_rps']:.1f} RPS")
        print(f"    mean={result['mean_ms']:.1f}ms  p95={result['p95_ms']:.1f}ms  max={result['max_ms']:.1f}ms")

await run_all_concurrency()

# 计算队列时间（p95 增长量）
if len(concurrency_results) >= 2:
    baseline_p95 = concurrency_results[0]["p95_ms"]
    print("\n[队列延迟分析]")
    print(f"  基线 p95 (并发=1): {baseline_p95:.1f}ms")
    for r in concurrency_results[1:]:
        queue_overhead = r["p95_ms"] - baseline_p95
        print(f"  并发={r['concurrency']}: p95={r['p95_ms']:.1f}ms  队列开销=+{queue_overhead:.1f}ms")

In [ ]:
# ============================================================
# CELL 8: [Nginx] 生产环境 Nginx 配置分析（静态分析）
# 由于开发环境无 Nginx，通过配置文件分析各 location 的超时和策略
# ============================================================

print("=" * 60)
print("[API-1] Nginx 配置分析（生产环境）")
print("=" * 60)

nginx_config_path = "/Users/dantevonalcatraz/Development/procurement-agents/xiaocai-platform/nginx/xiaocai.conf"

nginx_analysis = {
    "locations": [
        {
            "path": "/",
            "upstream": "xiaocai_app (port 3000)",
            "proxy_read_timeout": "60s",
            "proxy_buffering": "default (on)",
            "rate_limit": "无",
            "note": "前端静态资源"
        },
        {
            "path": "/api/",
            "upstream": "xiaocai_api (port 8001)",
            "proxy_read_timeout": "120s",
            "proxy_buffering": "default (on)",
            "rate_limit": "10r/s burst=20",
            "note": "REST API 端点"
        },
        {
            "path": "/graphql",
            "upstream": "xiaocai_api (port 8001)",
            "proxy_read_timeout": "300s",
            "proxy_buffering": "off (SSE 专用)",
            "rate_limit": "5r/s burst=10",
            "note": "GraphQL + SSE 流，关闭 buffering 保证实时推流"
        },
        {
            "path": "/graphql/subscriptions",
            "upstream": "xiaocai_api (port 8001)",
            "proxy_read_timeout": "7d",
            "proxy_buffering": "N/A (WebSocket)",
            "rate_limit": "无",
            "note": "WebSocket 长连接"
        },
        {
            "path": "/api/health",
            "upstream": "xiaocai_api (port 8001)",
            "proxy_read_timeout": "default",
            "proxy_buffering": "default",
            "rate_limit": "无 (access_log off)",
            "note": "健康检查专用，不记录日志"
        }
    ],
    "global_settings": {
        "upstream_keepalive": 32,
        "gzip_enabled": True,
        "ssl_protocols": "TLSv1.2 TLSv1.3",
        "rate_limit_zones": [
            {"zone": "api_limit",     "rate": "10r/s", "burst": 20},
            {"zone": "graphql_limit", "rate": "5r/s",  "burst": 10}
        ]
    }
}

print("\nNginx location 配置:")
print(f"  {'路径':<30} {'上游':<30} {'proxy_read_timeout':<20} {'限流'}")
print("  " + "-" * 100)
for loc in nginx_analysis["locations"]:
    print(f"  {loc['path']:<30} {loc['upstream']:<30} {loc['proxy_read_timeout']:<20} {loc['rate_limit']}")
    print(f"  {'':30} {loc['note']}")
    print()

print("\n关键设计点:")
print("  1. /graphql 专门关闭 proxy_buffering=off → SSE 数据立即透传，不等 buffer 满")
print("  2. 上游 keepalive=32 → 连接复用，减少 TCP 握手开销")
print("  3. API 限流 10r/s (burst=20) → 突发时平滑，单个 IP 峰值 30 QPS")
print("  4. GraphQL 限流 5r/s (burst=10) → AI 调用更保守")
print("  5. Nginx 本机延迟估算: ~0.1-0.5ms (kernel bypass loopback)")

In [ ]:
# ============================================================
# CELL 9: 综合可视化 — 4 个子图
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("API 网关层内部分析", fontsize=16, fontweight='bold', y=0.98)

COLORS = {
    "engine":  "#2ecc71",  # 绿色 = AI Engine 直接
    "gateway": "#3498db",  # 蓝色 = API 网关
    "graphql": "#9b59b6",  # 紫色 = GraphQL
    "cors":    "#e67e22",  # 橙色 = CORS
    "perf":    "#e74c3c",  # 红色 = 性能指标
}

# ── 子图 1: 路由层次对比 ──────────────────────────────────
ax1 = axes[0, 0]
ax1.set_title("[API-2] FastAPI 路由层次开销", fontweight='bold')

route_labels = list(routing_results.keys())
route_means  = [routing_results[k]["stats"].get("mean", 0) for k in route_labels
                if isinstance(routing_results[k], dict) and "stats" in routing_results[k]]
route_labels = [k for k in route_labels
                if isinstance(routing_results[k], dict) and "stats" in routing_results[k]]

if route_means:
    bar_colors = []
    for label in route_labels:
        if "Engine" in label:
            bar_colors.append(COLORS["engine"])
        else:
            bar_colors.append(COLORS["gateway"])
    bars = ax1.barh(range(len(route_labels)), route_means, color=bar_colors)
    ax1.set_yticks(range(len(route_labels)))
    ax1.set_yticklabels(route_labels, fontsize=9)
    ax1.set_xlabel("延迟 (ms)")
    for bar, val in zip(bars, route_means):
        ax1.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                 f"{val:.1f}ms", va='center', fontsize=8)
    ax1.axvline(x=route_means[0], color='red', linestyle='--', alpha=0.5,
                label=f"AI Engine 基线")
    ax1.legend(fontsize=8)
    patches = [
        mpatches.Patch(color=COLORS["engine"],  label="AI Engine 直连"),
        mpatches.Patch(color=COLORS["gateway"], label="API 网关"),
    ]
    ax1.legend(handles=patches, fontsize=8)

# ── 子图 2: GraphQL vs REST 对比 ──────────────────────────
ax2 = axes[0, 1]
ax2.set_title("[API-4] GraphQL vs REST 解析开销", fontweight='bold')

gql_labels = [k for k in graphql_results.keys() if not k.startswith("_")]
gql_means  = [graphql_results[k]["stats"].get("mean", 0) for k in gql_labels]

if gql_means:
    gql_colors = []
    for label in gql_labels:
        if "REST" in label:
            gql_colors.append(COLORS["engine"])
        elif "mutation" in label.lower():
            gql_colors.append(COLORS["perf"])
        else:
            gql_colors.append(COLORS["graphql"])
    bars = ax2.bar(range(len(gql_labels)), gql_means, color=gql_colors)
    ax2.set_xticks(range(len(gql_labels)))
    ax2.set_xticklabels(gql_labels, rotation=30, ha='right', fontsize=9)
    ax2.set_ylabel("延迟 (ms)")
    for bar, val in zip(bars, gql_means):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f"{val:.0f}ms", ha='center', va='bottom', fontsize=8)
    if gql_means:
        ax2.axhline(y=gql_means[0], color='green', linestyle='--', alpha=0.6,
                    label=f"REST 基线 {gql_means[0]:.1f}ms")
        ax2.legend(fontsize=8)

# ── 子图 3: 并发级别 vs 延迟 ───────────────────────────────
ax3 = axes[1, 0]
ax3.set_title("[API-6] 并发级别 vs 延迟（队列时间）", fontweight='bold')

if concurrency_results:
    conc_levels = [r["concurrency"] for r in concurrency_results]
    mean_ms     = [r["mean_ms"] for r in concurrency_results]
    p95_ms      = [r["p95_ms"] for r in concurrency_results]
    max_ms      = [r["max_ms"] for r in concurrency_results]

    ax3.plot(conc_levels, mean_ms, 'o-', color=COLORS["gateway"], label="mean", linewidth=2)
    ax3.plot(conc_levels, p95_ms,  's--', color=COLORS["cors"],    label="p95",  linewidth=2)
    ax3.plot(conc_levels, max_ms,  '^:', color=COLORS["perf"],    label="max",  linewidth=1)
    ax3.set_xlabel("并发数")
    ax3.set_ylabel("延迟 (ms)")
    ax3.set_xticks(conc_levels)
    ax3.legend(fontsize=10)
    ax3.grid(True, alpha=0.3)

    # 吞吐量双 Y 轴
    ax3_twin = ax3.twinx()
    rps_list = [r["throughput_rps"] for r in concurrency_results]
    ax3_twin.bar(conc_levels, rps_list, alpha=0.2, color='gray', label="RPS")
    ax3_twin.set_ylabel("吞吐量 (RPS)", color='gray')
    ax3_twin.tick_params(axis='y', labelcolor='gray')

# ── 子图 4: API 层时间分布瀑布 ────────────────────────────
ax4 = axes[1, 1]
ax4.set_title("[综合] API 网关层各环节开销估算", fontweight='bold')

# 构建各层时间估算
gateway_routing_mean = routing_results.get("API GW /health", {}).get("stats", {}).get("mean", 5.0)
gql_hello_mean       = graphql_results.get("GraphQL {hello}", {}).get("stats", {}).get("mean", 10.0)
rest_mean_val        = graphql_results.get("REST GET /health", {}).get("stats", {}).get("mean", 3.0)
hop_cost             = hop_results.get("_estimated_hop_ms", 1.0)
cors_options_mean    = cors_results.get("OPTIONS /graphql (CORS)", {}).get("mean", 5.0)

layers = [
    ("TCP 建连 (loopback)",     hop_cost * 0.3,   COLORS["engine"]),
    ("FastAPI 路由匹配",          rest_mean_val,    COLORS["gateway"]),
    ("CORS 中间件",               cors_options_mean * 0.1, COLORS["cors"]),
    ("GraphQL 解析校验",          gql_hello_mean - rest_mean_val, COLORS["graphql"]),
    ("API→Engine HTTP hop",     hop_cost,         COLORS["perf"]),
]
layers = [(l, max(v, 0.01), c) for l, v, c in layers]  # 确保非负

layer_names = [l[0] for l in layers]
layer_vals  = [l[1] for l in layers]
layer_cols  = [l[2] for l in layers]

bars = ax4.barh(range(len(layers)), layer_vals, color=layer_cols)
ax4.set_yticks(range(len(layers)))
ax4.set_yticklabels(layer_names, fontsize=9)
ax4.set_xlabel("开销 (ms)")
for bar, val in zip(bars, layer_vals):
    ax4.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
             f"{val:.1f}ms", va='center', fontsize=8)

total_overhead = sum(layer_vals)
ax4.set_title(f"[综合] API 网关层各环节开销估算\n总计: ~{total_overhead:.1f}ms",
              fontweight='bold')

plt.tight_layout()
ts = datetime.now().strftime("%Y%m%d_%H%M")
save_path = f"{OUTPUT_DIR}/api-gateway-internals_{ts}.png"
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"图表已保存: {save_path}")

In [ ]:
# ============================================================
# CELL 10: 汇总报告 + 保存
# ============================================================

print("=" * 60)
print("API 网关层分析汇总报告")
print("=" * 60)

# 各环节开销汇总
print("\n[各事件链节点 — 时间开销]")
print(f"  {'事件':<40} {'mean (ms)':>12} {'p95 (ms)':>12} {'说明'}")
print("  " + "-" * 90)

summary_rows = []

# 路由开销
for label in ["AI Engine /health", "API GW /health"]:
    if label in routing_results and routing_results[label].get("stats"):
        s = routing_results[label]["stats"]
        row = {"event": f"[API-2] {label}", "mean_ms": s.get("mean"), "p95_ms": s.get("p95"),
               "desc": routing_results[label].get("desc", "")}
        summary_rows.append(row)
        print(f"  {row['event']:<40} {str(s.get('mean')):>12} {str(s.get('p95')):>12}  {row['desc']}")

# GraphQL 开销
for label in ["REST GET /health", "GraphQL {hello}", "GraphQL __schema"]:
    if label in graphql_results and graphql_results[label].get("stats"):
        s = graphql_results[label]["stats"]
        row = {"event": f"[API-4] {label}", "mean_ms": s.get("mean"), "p95_ms": s.get("p95"),
               "desc": ""}
        summary_rows.append(row)
        print(f"  {row['event']:<40} {str(s.get('mean')):>12} {str(s.get('p95')):>12}")

# CORS 开销
for label in cors_results:
    s = cors_results[label]
    row = {"event": f"[API-3] {label}", "mean_ms": s.get("mean"), "p95_ms": s.get("p95"), "desc": ""}
    summary_rows.append(row)
    print(f"  {row['event']:<40} {str(s.get('mean')):>12} {str(s.get('p95')):>12}")

# 内部 hop
hop_ms = hop_results.get("_estimated_hop_ms", 0)
print(f"  {'[API-5] 内部 HTTP Hop (loopback)':<40} {str(round(hop_ms,2)):>12} {'N/A':>12}  估算值")

# 并发
print("\n[并发处理能力]")
print(f"  {'并发数':>8} {'成功/总数':>12} {'吞吐量 (RPS)':>14} {'mean (ms)':>10} {'p95 (ms)':>10}")
print("  " + "-" * 60)
for r in concurrency_results:
    print(f"  {r['concurrency']:>8} {r['success_count']:>6}/{r['total_requests']:<6} "
          f"{r['throughput_rps']:>14.1f} {r['mean_ms']:>10.1f} {r['p95_ms']:>10.1f}")

# 关键结论
print("\n[关键结论]")
gw_overhead = routing_results.get("_gateway_overhead_ms", "N/A")
gql_overhead = graphql_results.get("_gql_overhead_ms", "N/A")
print(f"  1. API 网关路由额外开销: {gw_overhead}ms")
print(f"  2. GraphQL 框架额外开销: {gql_overhead}ms")
print(f"  3. 内部 HTTP Hop 成本: ~{hop_ms:.2f}ms")
print(f"  4. SSE /graphql/stream 端点: 2次 hop（API→Engine），总转发开销 ~{hop_ms*2:.2f}ms")
print(f"  5. Nginx 在生产环境: SSE endpoint 设置 proxy_buffering=off，确保零延迟透传")

# 保存数据
import json
snapshot = {
    "timestamp": datetime.now().isoformat(),
    "notebook": "10-api-gateway-internals",
    "routing_results": {
        k: v["stats"] if isinstance(v, dict) and "stats" in v else v
        for k, v in routing_results.items()
    },
    "graphql_results": {
        k: v["stats"] if isinstance(v, dict) and "stats" in v else v
        for k, v in graphql_results.items()
    },
    "cors_results": cors_results,
    "hop_results": {
        k: v for k, v in hop_results.items()
        if isinstance(v, (int, float, str))
    },
    "concurrency_results": concurrency_results,
    "nginx_analysis": nginx_analysis,
}

ts = datetime.now().strftime("%Y%m%d_%H%M")
json_path = f"{OUTPUT_DIR}/api-gateway-internals_{ts}.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(snapshot, f, ensure_ascii=False, indent=2, default=str)

print(f"\n数据已保存: {json_path}")
print("\n10-api-gateway-internals.ipynb 执行完成")